# 3.6 Anwendung: Parametrische Analyse der Messbrücke

In Kapitel 3.5 haben wir die Funktion `solve_bridge(R4)` entwickelt, die
den Querstrom $I$ für einen einzelnen Widerstandswert berechnet. Jetzt
nutzen wir sie systematisch: Wir variieren $R_4$ über einen großen Bereich,
berechnen $I(R_4)$ und die Verlustleistung $P_B(R_4) = R_B \cdot I^2$ und
stellen die Ergebnisse grafisch dar. Das Ziel ist, den Abgleichwert
$R_4^*$ zu finden, bei dem $I = 0$ gilt, und ihn mit der analytischen
Formel aus Kapitel 3.5 zu vergleichen.

## Lernziele

* [ ] Sie können eine Parameterstudie mit einer `for`-Schleife über
  viele $R_4$-Werte durchführen und die Ergebnisse in Arrays speichern.
* [ ] Sie können $I(R_4)$ und $P_B(R_4)$ in einem Subplot mit
  gemeinsamer x-Achse darstellen.
* [ ] Sie können die Nullstelle von $I(R_4)$ numerisch mit
  `np.argmin` bestimmen und physikalisch erklären.
* [ ] Sie können das numerische Ergebnis mit einer analytischen Formel
  vergleichen und die Übereinstimmung prüfen.

## Vorbereitung: `solve_bridge` übernehmen

Wir übernehmen die Funktion und die festen Parameter aus Kapitel 3.5.
Die Koeffizientenmatrix ist dieselbe wie zuvor; nur $R_4$ wird als
Parameter übergeben:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

# --- Feste Parameter (als Notebook-Konstanten und Funktions-Defaults) ---
R1 = 100.   # Ohm
R2 = 100.   # Ohm
R3 = 100.   # Ohm
RB = 10.    # Ohm
U0 = 10.    # V

def solve_bridge(R4, R1=100., R2=100., R3=100., RB=10., U0=10.):
    """Berechnet den Querstrom I durch R_B für gegebenes R4.

    Parameter
    ----------
    R4 : float
        Variabler Messwiderstand in Ohm
    R1, R2, R3 : float
        Feste Brückenwiderstände in Ohm (Default: 100)
    RB : float
        Brückenwiderstand in Ohm (Default: 10)
    U0 : float
        Speisespannung in V (Default: 10)

    Rückgabe
    --------
    I : float
        Querstrom in Ampere
    """
    # Koeffizientenmatrix aus Kapitel 3.5 (Kirchhoffsche Regeln)
    A = np.array([
        [+1., -1.,  0., -1.,   0.,   0.],   # K1
        [ 0., +1., -1.,  0.,   0.,  -1.],   # K2
        [ 0.,  0.,  0., +1.,  -1.,  +1.],   # K3
        [ 0.,  R1,  R2,  0.,   0.,   0.],   # M1
        [ 0.,  0.,  0.,  R3,   R4,   0.],   # M2
        [ 0.,  R1,  0., -R3,   0.,  -RB],   # M3
    ])
    b = np.array([0., 0., 0., U0, U0, 0.])
    return np.linalg.solve(A, b)[5]

## Parameterstudie

Wir lassen $R_4$ von $1\,\Omega$ bis $2000\,\Omega$ laufen und rufen
`solve_bridge` für jeden Wert auf. Die Ergebnisse speichern wir
parallel in zwei Arrays:

In [ ]:
# R4-Werte und Ergebnis-Arrays vorbereiten
R4_werte = np.linspace(1., 2000., 500)
I_werte  = np.zeros(len(R4_werte))
PB_werte = np.zeros(len(R4_werte))

# --- Parameterstudie: ein LGS pro R4-Wert ---
for i, R4 in enumerate(R4_werte):
    I_i         = solve_bridge(R4)
    I_werte[i]  = I_i
    PB_werte[i] = RB * I_i**2   # Verlustleistung P_B = R_B * I^2

print(f'Minimaler Querstrom:      {I_werte.min()*1000:.4f} mA')
print(f'Maximaler Querstrom:      {I_werte.max()*1000:.4f} mA')
print(f'Maximale Verlustleistung: {PB_werte.max()*1000:.4f} mW')

## Visualisierung

Wir zeigen $I(R_4)$ und $P_B(R_4)$ übereinander. `sharex=True` koppelt
die x-Achsen beider Subplots: Ein Zoom im oberen Diagramm wirkt sich
automatisch auf das untere aus, und die x-Achsenbeschriftung erscheint
nur einmal unten:

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(9, 7), sharex=True)

# --- Oberes Diagramm: Querstrom ---
ax[0].plot(R4_werte, I_werte * 1000,   # Umrechnung A -> mA
           linewidth=2)
ax[0].axhline(0, color='gray', linestyle='dashed', linewidth=1)
ax[0].set_ylabel('Querstrom $I$ in mA')
ax[0].set_title('Messbrücke: Querstrom und Verlustleistung als Funktion von $R_4$')
ax[0].grid(True)

# --- Unteres Diagramm: Verlustleistung ---
ax[1].plot(R4_werte, PB_werte * 1000,  # Umrechnung W -> mW
           linewidth=2)
ax[1].set_xlabel('$R_4$ in $\\Omega$')
ax[1].set_ylabel('Verlustleistung $P_B$ in mW')
ax[1].grid(True)

plt.tight_layout()
plt.show()

### Mini-Übung 1

Sehen Sie sich das Diagramm an und beantworten Sie folgende Fragen,
bevor Sie weiteren Code ausführen.

1. Bei welchem $R_4$-Wert schätzen Sie die Nullstelle von $I$ aus dem
   Diagramm ab?
2. Für welche $R_4$-Werte ist $I$ positiv, für welche negativ? Erklären
   Sie das Vorzeichen physikalisch: Was bedeutet ein Vorzeichenwechsel
   für die Stromrichtung durch $R_B$?
3. $P_B = R_B \cdot I^2$ hat bei $R_4^*$ ein Minimum. Warum ist dieses
   Minimum gleich null, und warum kann $P_B$ nicht negativ werden?
   Skizzieren Sie den qualitativen Verlauf von $P_B$ in der Nähe
   der Nullstelle, bevor Sie nachsehen.

## Nullstelle bestimmen und Abgleichbedingung prüfen

Wir suchen den Index des betragskleinsten Querstroms mit `np.argmin`
und vergleichen den numerisch gefundenen Wert mit der analytischen
Abgleichbedingung der Wheatstone-Brücke:

$$\frac{R_1}{R_2} = \frac{R_3}{R_4^*}
\quad\Rightarrow\quad
R_4^* = \frac{R_2 \cdot R_3}{R_1}$$

In [ ]:
# --- Nullstelle numerisch bestimmen ---
# np.argmin(np.abs(...)) findet den Index des betragskleinsten Eintrags
idx_null = np.argmin(np.abs(I_werte))
R4_null  = R4_werte[idx_null]

print(f'Numerisch gefunden:            R4* ≈ {R4_null:.1f} Ω')
print(f'I bei R4*: {I_werte[idx_null]*1e9:.2f} nA  (numerisch ≈ 0)')

# --- Analytische Abgleichbedingung ---
R4_analytisch = R2 * R3 / R1
print(f'Analytische Abgleichbedingung: R4* = {R4_analytisch:.1f} Ω')
print(f'Übereinstimmung: {np.isclose(R4_analytisch, R4_null, atol=5.)}')

# --- Nullstelle im Diagramm markieren ---
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(9, 7), sharex=True)

ax[0].plot(R4_werte, I_werte * 1000, linewidth=2,
           label='Querstrom $I$')
ax[0].axhline(0, color='gray', linestyle='dashed', linewidth=1)
ax[0].axvline(R4_null, color='tab:red', linestyle='dashed', linewidth=1.5,
              label=f'$R_4^* = {R4_null:.0f}\\,\\Omega$')
ax[0].scatter([R4_null], [I_werte[idx_null]*1000],
              color='tab:red', s=80, zorder=5)
ax[0].set_ylabel('Querstrom $I$ in mA')
ax[0].set_title('Messbrücke: Nullstelle bei $R_4^*$')
ax[0].legend()
ax[0].grid(True)

ax[1].plot(R4_werte, PB_werte * 1000, linewidth=2,
           label='Verlustleistung $P_B$')
ax[1].axvline(R4_null, color='tab:red', linestyle='dashed', linewidth=1.5,
              label=f'$R_4^* = {R4_null:.0f}\\,\\Omega$')
ax[1].scatter([R4_null], [PB_werte[idx_null]*1000],
              color='tab:red', s=80, zorder=5)
ax[1].set_xlabel('$R_4$ in $\\Omega$')
ax[1].set_ylabel('Verlustleistung $P_B$ in mW')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

Die kleine Abweichung zwischen numerischem und analytischem Wert entsteht,
weil `np.linspace` die Nullstelle nicht exakt trifft, sondern nur
annähert. Je feiner das Gitter, desto kleiner die Abweichung.

### Mini-Übung 2

In der Praxis ist $R_3$ der einstellbare Referenzwiderstand und $R_4$
der unbekannte Messwiderstand. Die Brücke wird abgeglichen, indem man
$R_3$ so lange variiert, bis $I = 0$ gilt. Aus dem Abgleichwert $R_3^*$
lässt sich dann $R_4$ bestimmen.

1. Schreiben Sie eine Funktion `solve_bridge_R3(R3_var)`, die bei
   festem $R_4 = 150\,\Omega$ den Querstrom als Funktion von $R_3$
   berechnet. Die Koeffizientenmatrix hat dieselbe Struktur wie in
   `solve_bridge`, nur $R_3$ und $R_4$ tauschen die Rollen.
2. Führen Sie eine Parameterstudie für
   $R_3 \in [1\,\Omega, 400\,\Omega]$ durch und stellen Sie $I(R_3)$
   grafisch dar.
3. Bestimmen Sie den Abgleichwert $R_3^*$ numerisch mit
   `np.argmin(np.abs(...))`.
4. Überprüfen Sie mit der analytischen Abgleichbedingung
   $R_3^* = R_1 \cdot R_4 / R_2$, ob Ihr Ergebnis stimmt.

In [ ]:
# Hier Ihren Code eingeben

## Zusammenfassung und Ausblick

Wir haben `solve_bridge` in einer Schleife über 500 $R_4$-Werte
aufgerufen und die Ergebnisse als NumPy-Arrays gespeichert. Der Subplot
mit `sharex=True` erlaubt den direkten Vergleich von $I(R_4)$ und
$P_B(R_4)$. Die Nullstelle bei $R_4^* = 100\,\Omega$, numerisch über
`np.argmin` gefunden, bestätigt das Wheatstonesche Abgleichprinzip und
stimmt mit der analytischen Formel überein.

Im nächsten Kapitel wechseln wir die Perspektive: Statt eines
physikalischen Problems untersuchen wir, wie die Rechenzeit von
`np.linalg.solve` mit der Systemgröße $n$ wächst, und überprüfen die
theoretische $O(n^3)$-Skalierung der LU-Zerlegung experimentell.